In [1]:
import math
from pathlib import Path
from typing import Tuple

import h5py as h5
import numpy as np


In [2]:
def tile_volume_h5(
    in_path: Path,
    in_key: str,
    out_dir: Path,
    tile_size: Tuple[int, int, int] = (128, 128, 128),   # (Z,Y,X)
    stride: Tuple[int, int, int] = (128, 128, 128),      # (Z,Y,X)
    out_key: str = "volume",
    compression: str = "gzip",
    compression_opts: int = 4,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    tz, ty, tx = map(int, tile_size)
    sz, sy, sx = map(int, stride)

    with h5.File(in_path, "r") as F:
        ds = F[in_key]
        Z, Y, X = ds.shape
        dtype = ds.dtype

        tile_id = 0
        for z0 in range(0, Z, sz):
            for y0 in range(0, Y, sy):
                for x0 in range(0, X, sx):
                    z1 = min(z0 + tz, Z)
                    y1 = min(y0 + ty, Y)
                    x1 = min(x0 + tx, X)

                    # skip incomplete edge tiles (optional)
                    if (z1 - z0) != tz or (y1 - y0) != ty or (x1 - x0) != tx:
                        continue

                    tile = ds[z0:z1, y0:y1, x0:x1]
                    name = in_path.name.split(".", 1)[0]
                    out_path = out_dir / f"{name}.sub-{tile_id:03d}.vol.h5"
                    with h5.File(out_path, "w") as G:
                        dso = G.create_dataset(
                            out_key,
                            data=tile,
                            dtype=dtype,
                            chunks=(min(tz, 128), min(ty, 128), min(tx, 128)),
                            compression=compression,
                            compression_opts=compression_opts,
                        )

                        # origin indices for reassembly
                        dso.attrs["origin_zyx"] = (z0, y0, x0)
                        dso.attrs["tile_size_zyx"] = (tz, ty, tx)
                        dso.attrs["stride_zyx"] = (sz, sy, sx)
                        dso.attrs["source_shape_zyx"] = (Z, Y, X)
                        dso.attrs["source_path"] = str(in_path)
                        dso.attrs["source_key"] = str(in_key)

                    tile_id += 1

    print(f"Wrote {tile_id} tiles to {out_dir}")


In [3]:
tile_volume_h5(
        in_path=Path("../data/original/285_01_HR_.vol.h5"),
        in_key="volume",
        out_dir=Path("../data/original/subvolumes"),
        tile_size=(1024, 1024, 1024),
        stride=(768,768,768),  
    )

Wrote 8 tiles to ../data/original/subvolumes
